<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller Hands-On: Una Ciudad en Movimiento, con Rigor y con Ética 🚦
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Taller Práctico Evaluativo · Módulo 04
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Estadistica%20analisis%20y%20representacion%20de%20datos/homeworks/04_NoParametrica_Regresion_Etica_Hands_On.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Este taller consolida el **Módulo 04: No Paramétrica, Regresión y Ética** ([README del módulo](../04%20-%20No%20Parametrica%20Regresion%20y%20Etica/README.md)) con un problema **nuevo**: el **observatorio de movilidad de una ciudad intermedia ficticia** (tiempos de viaje por ruta, flujo vehicular, entregas a domicilio, un registro de salud y las decisiones de un modelo de crédito).

Al terminar serás capaz de:

1. **Comparar grupos sin suponer normalidad:** rangos con empates, $U$ de Mann-Whitney (aproximación normal con varianza corregida y prueba de permutación), $H$ de Kruskal-Wallis y un tamaño del efecto interpretable, y ver por qué un solo dato mal digitado arruina a la $t$ pero no a los rangos.
2. **Ajustar e interpretar una regresión simple** desde cero: descomposición $SCT = SCR + SCE$, error estándar e intervalo de la pendiente, **banda de confianza frente a banda de predicción**, y la diferencia entre **Pearson** y **Spearman**.
3. **Construir y diagnosticar una regresión múltiple** en forma matricial: errores estándar, matriz sombrero y distancia de Cook, **VIF**, sesgo por **variable omitida** y $F$ parcial.
4. **Usar los datos con responsabilidad:** medir el $k$-anonimato y suprimir lo que reidentifica, calibrar el ruido de **privacidad diferencial**, controlar la **multiplicidad** (Bonferroni, Holm, Benjamini-Hochberg) y auditar la **equidad** de un modelo por grupos.

> 📚 **Repasa si lo necesitas:** [No paramétrica](../04%20-%20No%20Parametrica%20Regresion%20y%20Etica/00_Estadistica_No_Parametrica_Kruskal_Wallis_y_Mann_Whitney.ipynb) · [Regresión simple y correlación](../04%20-%20No%20Parametrica%20Regresion%20y%20Etica/01_Regresion_Lineal_Simple_y_Correlacion.ipynb) · [Regresión múltiple y diagnóstico](../04%20-%20No%20Parametrica%20Regresion%20y%20Etica/02_Regresion_Multiple_y_Diagnostico.ipynb) · [Ética en el uso de datos](../04%20-%20No%20Parametrica%20Regresion%20y%20Etica/03_Etica_en_el_Uso_de_Datos.ipynb).

---
## Instrucciones 📋

1. **Resuelve de forma autónoma:** en cada reto completa la celda de código marcada con `# TODO:`. Este cuaderno **no trae soluciones**.
2. **Autoverificación:** debajo de cada reto hay una celda con `assert`. Si tu solución es correcta imprime ✅; si no, el mensaje te dice qué revisar. No la modifiques.
3. **Puntaje:** cada reto indica su puntaje sugerido; el total del taller suma **100 puntos**.
4. Antes de entregar usa *Restart & Run All*: el cuaderno debe correr de principio a fin, sin errores y **sin `# TODO` pendientes**.
5. Todo funciona **sin internet y sin archivos externos**: los datos son **sintéticos e inventados**, con semilla fija (los resultados describen el mecanismo, no una ciudad real). Bibliotecas: `numpy`, `pandas`, `scipy` y `statsmodels` (esta última solo para contrastar tus resultados). Todo el cuaderno corre en pocos segundos.

In [ ]:
# ============================================================
# Preparación (no modificar): imports y datos sintéticos
# ============================================================
import math

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor

SEMILLA = 11


def generar_viajes(semilla=SEMILLA):
    """Tiempos de viaje (minutos enteros) de tres rutas ficticias; la B es algo más lenta."""
    rng = np.random.default_rng(semilla)
    n, med = {"A": 40, "B": 45, "C": 35}, {"A": 24, "B": 26, "C": 24.5}
    filas = []
    for r in "ABC":
        t = np.maximum(5, np.round(rng.lognormal(np.log(med[r]), 0.28, n[r])))
        filas += [(r, v) for v in t]
    return pd.DataFrame(filas, columns=["ruta", "minutos"])


def generar_trafico(semilla=SEMILLA, n=60):
    """Flujo (centenas de vehículos/h), tiempo de recorrido (min) y consumo de combustible (monótono, convexo)."""
    rng = np.random.default_rng(semilla + 1)
    f = np.round(rng.uniform(2, 20, n), 1)
    t = 8 + 1.9 * f + rng.normal(0, 4, n)
    c = 2 * np.exp(0.2 * f) * np.exp(rng.normal(0, 0.25, n))
    return pd.DataFrame({"flujo": f, "tiempo": np.round(t, 1), "consumo": np.round(c, 1)})


def generar_entregas(semilla=SEMILLA, n=150):
    """Entregas a domicilio: la fila 0 es una entrega lejana e inusualmente rápida (apalancamiento)."""
    rng = np.random.default_rng(semilla + 2)
    dist = np.round(rng.uniform(0.5, 12, n), 2)
    cuad = np.round(12 * dist + rng.normal(0, 3, n))
    llu = np.round(rng.exponential(3, n), 1)
    ped = np.maximum(0, np.round(4 + 0.8 * llu + rng.normal(0, 2, n)))
    zona = rng.choice(["centro", "norte", "sur"], n)
    ef = {"centro": 0.0, "norte": 3.0, "sur": 1.5}
    t = 6 + 2.5 * dist + 1.2 * ped + 0.4 * llu + np.array([ef[z] for z in zona]) + rng.normal(0, 3, n)
    dist[0], cuad[0], ped[0], llu[0], zona[0] = 24.0, 291.0, 5.0, 1.0, "centro"
    t[0] = 6 + 2.5 * 24 + 1.2 * 5 + 0.4 - 50
    return pd.DataFrame({"distancia": dist, "cuadras": cuad, "pedidos": ped, "lluvia": llu, "zona": zona, "tiempo": np.round(t, 1)})


def generar_registro(semilla=SEMILLA, n=2500):
    """Registro de salud ficticio: cuasi-identificadores (edad, municipio, ocupación) y un diagnóstico sensible."""
    rng = np.random.default_rng(semilla + 3)
    edad = np.clip(np.round(rng.normal(42, 15, n)), 18, 85).astype(int)
    mun = rng.choice(["Tunja", "Duitama", "Sogamoso", "Paipa", "Chiquinquirá", "Villa de Leyva", "Samacá", "Ramiriquí"], n,
                     p=[.28, .2, .18, .1, .08, .06, .06, .04])
    ocu = rng.choice(["agricultor", "comerciante", "docente", "conductor", "estudiante", "salud", "otro"], n,
                     p=[.2, .2, .1, .12, .18, .08, .12])
    dx = np.where(rng.random(n) < 0.12, "positivo", "negativo")
    return pd.DataFrame({"edad": edad, "municipio": mun, "ocupacion": ocu, "diagnostico": dx})


def generar_pvalores(semilla=SEMILLA):
    """20 valores p de una exploración: 3 efectos fuertes, 3 moderados y 14 hipótesis nulas (p uniforme)."""
    rng = np.random.default_rng(semilla + 4)
    p = np.concatenate([10 ** rng.uniform(-6, -4, 3), rng.uniform(0.002, 0.03, 3), rng.uniform(0, 1, 14)])
    return rng.permutation(p)


def generar_decisiones(semilla=SEMILLA, n=4000):
    """Decisiones históricas de un modelo de crédito: grupo (A, B), pago real (1 = pagó) y aprobación (1 = aprobado)."""
    rng = np.random.default_rng(semilla + 5)
    g = np.where(rng.random(n) < 0.4, "B", "A")
    y = (rng.random(n) < np.where(g == "A", 0.70, 0.66)).astype(int)
    pa = np.where(y == 1, np.where(g == "A", 0.85, 0.62), np.where(g == "A", 0.30, 0.18))
    return pd.DataFrame({"grupo": g, "pago": y, "aprobado": (rng.random(n) < pa).astype(int)})


print("Listo: viajes", generar_viajes().shape, "| tráfico", generar_trafico().shape, "| entregas", generar_entregas().shape,
      "| registro", generar_registro().shape, "| decisiones", generar_decisiones().shape)

---
## Reto 1: Rutas, rangos y un cronómetro mal digitado 🚌

**Contexto:** la alcaldía compara los **tiempos de viaje en minutos** de tres rutas (`viajes`, columnas `ruta` y `minutos`). Los tiempos son enteros, así que hay **muchos empates**, y la distribución tiene cola derecha. Para comparar la ruta A con la B usarás rangos. Después simularás un error de digitación en el viaje más largo de A.

**Enunciado**
1. `rangos(v)`: rangos $1,\dots,n$ de un vector, donde los empates reciben el **promedio** de los rangos que ocuparían (sin usar `rankdata`).
2. `mann_whitney(x, y)` → `(U, z, p)`: $U_x = R_x - \frac{n_x(n_x+1)}{2}$ con $R_x$ la suma de rangos de $x$ en la muestra conjunta; $z = \dfrac{U_x - n_xn_y/2}{\sigma}$ con $\sigma^2 = \dfrac{n_xn_y}{12}\left[(N+1) - \dfrac{\sum (t^3 - t)}{N(N-1)}\right]$ ($t$ = tamaño de cada grupo de empatados, $N = n_x + n_y$) y $p$ **bilateral** normal, **sin** corrección por continuidad.
3. `prob_sup` $= U_x/(n_xn_y)$ (probabilidad de que un viaje de A supere a uno de B, con los empates contando $\tfrac12$) y `r_biserial` $= 2\,\texttt{prob\_sup} - 1$.
4. `p_perm`: valor $p$ bilateral por **permutación** con `B = 5000` reordenamientos de las etiquetas (usa `np.random.default_rng(0)`; cuenta los $|U^{\ast}-\mu_U|\ge|U_x-\mu_U|$ y aplica $(\text{cuenta}+1)/(B+1)$).
5. **Error de digitación:** `x_err` es `x` con su **máximo multiplicado por 100** y `x_err_mil` lo multiplica por 1000. Calcula `p_t_limpio` y `p_t_err` (prueba $t$ de Welch, `stats.ttest_ind(..., equal_var=False)`) y `U_err`, `U_err_mil` (tu $U$ con cada versión).
6. `kruskal_wallis(grupos)` → `H` con **corrección por empates** ($H=\dfrac{\frac{12}{N(N+1)}\sum_i R_i^2/n_i - 3(N+1)}{1 - \sum(t^3-t)/(N^3-N)}$), `p_kw` (cola superior de $\chi^2_{k-1}$) y `eps2` $= H/(N-1)$ (tamaño del efecto $\varepsilon^2$).

**Criterios de aceptación:** `rangos`, $U$, $p$ y $H$ coinciden con SciPy; `p_perm` queda cerca de `p`; con el error de digitación la $t$ **deja de ser significativa** pero Mann-Whitney **no**; y `U_err == U_err_mil` (a los rangos no les importa *cuánto* más grande sea el valor extremo).

**Puntaje sugerido:** 22 puntos.

In [ ]:
viajes = generar_viajes()
x = viajes.loc[viajes["ruta"] == "A", "minutos"].to_numpy()
y = viajes.loc[viajes["ruta"] == "B", "minutos"].to_numpy()
grupos = [viajes.loc[viajes["ruta"] == r, "minutos"].to_numpy() for r in "ABC"]


def rangos(v):
    # TODO: rangos con empates promediados (el menor valor tiene rango 1)
    return None


def mann_whitney(x, y):
    # TODO: devuelve (U_x, z, p) con corrección de empates en la varianza y p bilateral sin corrección por continuidad
    return None


def kruskal_wallis(grupos):
    # TODO: devuelve H con corrección por empates
    return None


# TODO: completa
U_x, z, p_asint = None, None, None
prob_sup = None         # U_x / (nx * ny)
r_biserial = None       # 2 * prob_sup - 1

B = 5000
p_perm = None           # p bilateral por permutación (rng = np.random.default_rng(0))

x_err = None            # x con su máximo multiplicado por 100
x_err_mil = None        # x con su máximo multiplicado por 1000
p_t_limpio, p_t_err = None, None   # Welch con x y con x_err
U_err, U_err_mil = None, None      # tu U con x_err y con x_err_mil

H = None
p_kw = None             # cola superior de chi-cuadrado con k - 1 gl
eps2 = None             # H / (N - 1)

In [ ]:
# Autoverificación del Reto 1 (no modificar)
_r = rangos(np.array([3.0, 1.0, 3.0, 2.0]))
assert _r is not None and np.allclose(_r, [3.5, 1.0, 3.5, 2.0]), "rangos([3, 1, 3, 2]) debe dar [3.5, 1, 3.5, 2]: los dos 3 empatan en las posiciones 3 y 4 y reciben el promedio 3.5."
_todos = np.concatenate([x, y])
assert np.allclose(rangos(_todos), stats.rankdata(_todos)), "rangos debe coincidir con scipy.stats.rankdata (promedio en empates) sobre la muestra conjunta."
_sc = stats.mannwhitneyu(x, y, method="asymptotic", use_continuity=False)
_Ux = stats.rankdata(_todos)[:len(x)].sum() - len(x) * (len(x) + 1) / 2
assert U_x is not None and abs(U_x - _Ux) < 1e-9, "U_x = (suma de rangos de x) - nx(nx+1)/2, con los rangos de la muestra CONJUNTA."
assert abs(p_asint - _sc.pvalue) < 1e-9, "El p asintótico debe incluir la corrección por empates en la varianza (sin corrección por continuidad) y ser bilateral: 2 * norm.sf(|z|)."
_t = np.unique(_todos, return_counts=True)[1]
_N = len(_todos)
_s2 = len(x) * len(y) / 12 * ((_N + 1) - (_t ** 3 - _t).sum() / (_N * (_N - 1)))
assert abs(z - (U_x - len(x) * len(y) / 2) / math.sqrt(_s2)) < 1e-9, "z = (U - nx*ny/2) / sigma, con sigma^2 corregida por empates."
assert abs(prob_sup - _Ux / (len(x) * len(y))) < 1e-12 and abs(r_biserial - (2 * prob_sup - 1)) < 1e-12 and 0 < prob_sup < 0.5, "prob_sup = U_x/(nx*ny) y r_biserial = 2*prob_sup - 1. Como A es más rápida, prob_sup debe ser menor que 0.5."
assert p_perm is not None and 1 / (B + 1) <= p_perm <= 1 and abs(p_perm - p_asint) < 0.03, "p_perm debe estar cerca del p asintótico (diferencia < 0.03): permuta las etiquetas B veces, recalcula U y cuenta |U* - nx*ny/2| >= |U_x - nx*ny/2|; usa (cuenta + 1) / (B + 1)."
assert x_err is not None and abs(x_err.max() - 100 * x.max()) < 1e-9 and abs(x_err_mil.max() - 1000 * x.max()) < 1e-9 and (x_err != x).sum() == 1, "x_err y x_err_mil deben ser copias de x con SOLO el máximo multiplicado por 100 y por 1000 (no modifiques x)."
assert abs(p_t_limpio - stats.ttest_ind(x, y, equal_var=False).pvalue) < 1e-12 and abs(p_t_err - stats.ttest_ind(x_err, y, equal_var=False).pvalue) < 1e-12, "Usa la prueba t de Welch: stats.ttest_ind(a, b, equal_var=False).pvalue."
assert p_t_limpio < 0.05 < p_t_err, f"Con datos limpios la t detecta la diferencia (p = {p_t_limpio:.4f}); con el error de digitación debe dejar de hacerlo."
_pe = stats.mannwhitneyu(x_err, y, method="asymptotic", use_continuity=False).pvalue
assert abs(mann_whitney(x_err, y)[2] - _pe) < 1e-9 and _pe < 0.05, "Con el error de digitación Mann-Whitney sigue detectando la diferencia (p < 0.05): el valor extremo solo aporta su rango."
assert U_err is not None and abs(U_err - U_err_mil) < 1e-9, "U_err debe ser igual a U_err_mil: el máximo ya era el rango más alto, no importa cuánto lo multipliques."
_ks = stats.kruskal(*grupos)
assert H is not None and abs(H - _ks.statistic) < 1e-9, "H debe llevar la corrección por empates: divide por 1 - sum(t^3 - t)/(N^3 - N)."
assert abs(p_kw - _ks.pvalue) < 1e-9, "p_kw = stats.chi2.sf(H, k - 1) con k = 3 grupos."
assert abs(eps2 - _ks.statistic / (sum(len(g) for g in grupos) - 1)) < 1e-12, "eps2 = H / (N - 1), con N = 120 viajes (suma de los tres grupos)."
print(f"✅ Reto 1 superado: U = {U_x:.1f} (P(A>B) = {prob_sup:.3f}) | p asintótico {p_asint:.4f}, permutación {p_perm:.4f} | Welch pasa de p = {p_t_limpio:.4f} a {p_t_err:.3f} con el error, MW queda en {_pe:.4f} | H = {H:.2f} (p = {p_kw:.4f}, ε² = {eps2:.3f})")

---
## Reto 2: Flujo vehicular y tiempo de recorrido, de la recta a la predicción 📈

**Contexto:** `trafico` trae 60 mediciones de `flujo` (centenas de vehículos por hora), `tiempo` de recorrido (minutos) y `consumo` de combustible. Vas a ajustar $\text{tiempo} = \beta_0 + \beta_1\,\text{flujo} + \varepsilon$ **desde cero**, a decir con qué incertidumbre, y a mostrar por qué un coeficiente de correlación puede subestimar una relación fuerte pero curva.

**Enunciado**
1. `mco_simple(x, y)` → `(b0, b1)` por mínimos cuadrados (fórmulas con sumas de cuadrados y productos cruzados). Con ellos: `sct` (suma total de cuadrados), `scr` (de regresión, $\sum(\hat y_i-\bar y)^2$), `sce` (de residuos) y `r2`.
2. Con $s^2=SCE/(n-2)$: `se_b1`, e `ic_b1` (tupla, 95 %, cuantil $t_{n-2}$).
3. Para `X0 = 15` centenas de vehículos/h: `ic_media` (IC del **tiempo medio**) e `ic_pred` (intervalo de **predicción** de un recorrido nuevo), ambos de 95 %, y `razon_anchos` = ancho de `ic_pred` / ancho de `ic_media`.
4. Correlación: `r` de Pearson entre `flujo` y `tiempo` con `ic_r` (95 %, transformación de Fisher: $z=\operatorname{arctanh}(r)$, error estándar $1/\sqrt{n-3}$); además `r_c` (Pearson) y `rho_c` (Spearman = Pearson de los rangos; usa `stats.rankdata`) entre `flujo` y `consumo`.

**Criterios de aceptación:** todo coincide con `statsmodels`/SciPy (tolerancia $10^{-8}$); $SCT = SCR + SCE$; el intervalo de predicción **contiene** al de la media y es varias veces más ancho; y `rho_c > r_c`: la relación entre flujo y consumo es **monótona pero no lineal**.

**Puntaje sugerido:** 18 puntos.

In [ ]:
trafico = generar_trafico()
fl, ti, co = trafico["flujo"].to_numpy(), trafico["tiempo"].to_numpy(), trafico["consumo"].to_numpy()
n2, X0 = len(fl), 15.0


def mco_simple(x, y):
    # TODO: devuelve (b0, b1) con b1 = Sxy / Sxx y b0 = ybar - b1 * xbar
    return None


# TODO: completa
b0, b1 = None, None
sct = scr = sce = r2 = None
se_b1 = None
ic_b1 = None             # (inferior, superior)
ic_media = None          # IC 95 % del tiempo medio en X0
ic_pred = None           # intervalo de predicción 95 % en X0
razon_anchos = None
r = None
ic_r = None              # (inferior, superior) con Fisher
r_c = rho_c = None       # flujo vs consumo: Pearson y Spearman

In [ ]:
# Autoverificación del Reto 2 (no modificar)
_m = sm.OLS(ti, sm.add_constant(fl)).fit()
_f = _m.get_prediction(np.array([[1.0, X0]])).summary_frame(alpha=0.05)
assert b0 is not None and abs(b0 - _m.params[0]) < 1e-8 and abs(b1 - _m.params[1]) < 1e-8, "b1 = Σ(x - x̄)(y - ȳ) / Σ(x - x̄)²  y  b0 = ȳ - b1·x̄."
assert sct is not None and abs(sct - _m.centered_tss) < 1e-8 and abs(sce - _m.ssr) < 1e-8, "sct = Σ(y - ȳ)²;  sce = Σ(y - ŷ)²."
assert abs(scr - _m.ess) < 1e-8 and abs(sct - (scr + sce)) < 1e-8, "scr = Σ(ŷ - ȳ)² y debe cumplirse SCT = SCR + SCE."
assert abs(r2 - _m.rsquared) < 1e-10, "r2 = SCR / SCT."
assert se_b1 is not None and abs(se_b1 - _m.bse[1]) < 1e-8, "se_b1 = sqrt(s² / Sxx) con s² = SCE / (n - 2)."
assert ic_b1 is not None and np.allclose(ic_b1, _m.conf_int()[1], atol=1e-8), "ic_b1 = b1 ± t_{0.975, n-2} · se_b1."
assert ic_media is not None and np.allclose(ic_media, [_f["mean_ci_lower"][0], _f["mean_ci_upper"][0]], atol=1e-8), "IC de la media: ŷ0 ± t · sqrt(s²·(1/n + (x0 - x̄)²/Sxx))."
assert ic_pred is not None and np.allclose(ic_pred, [_f["obs_ci_lower"][0], _f["obs_ci_upper"][0]], atol=1e-8), "Intervalo de predicción: ŷ0 ± t · sqrt(s²·(1 + 1/n + (x0 - x̄)²/Sxx)); ¡lleva el 1 extra!"
assert ic_pred[0] < ic_media[0] < ic_media[1] < ic_pred[1] and razon_anchos > 3, "El intervalo de predicción debe contener al de la media y ser más de 3 veces más ancho."
assert abs(razon_anchos - (ic_pred[1] - ic_pred[0]) / (ic_media[1] - ic_media[0])) < 1e-12, "razon_anchos = ancho de ic_pred / ancho de ic_media."
assert r is not None and abs(r - stats.pearsonr(fl, ti)[0]) < 1e-10, "r = Sxy / sqrt(Sxx·Syy)."
_z, _e = np.arctanh(stats.pearsonr(fl, ti)[0]), 1 / np.sqrt(len(fl) - 3)
assert ic_r is not None and np.allclose(ic_r, np.tanh([_z - stats.norm.ppf(0.975) * _e, _z + stats.norm.ppf(0.975) * _e]), atol=1e-8), "ic_r: tanh(arctanh(r) ± 1.96/sqrt(n - 3)), es decir, se calcula en la escala z de Fisher y se regresa con tanh."
assert r_c is not None and abs(r_c - stats.pearsonr(fl, co)[0]) < 1e-10 and abs(rho_c - stats.spearmanr(fl, co)[0]) < 1e-10, "r_c es Pearson y rho_c es Pearson de los rangos (stats.rankdata) para flujo vs consumo."
assert rho_c > r_c + 0.03, "Para una relación monótona pero curva, Spearman debe superar a Pearson."
print(f"✅ Reto 2 superado: tiempo = {b0:.2f} + {b1:.3f}·flujo (R² = {r2:.3f}) | IC95 pendiente ({ic_b1[0]:.3f}, {ic_b1[1]:.3f}) | en X0 la predicción es {razon_anchos:.1f}× más ancha que la de la media | Pearson {r_c:.3f} < Spearman {rho_c:.3f} en consumo")

---
## Reto 3: Domicilios, colinealidad y la entrega que no encaja 🛵

**Contexto:** `entregas` tiene 150 domicilios con `distancia` (km), `cuadras` (que casi duplica la información de la distancia), `pedidos` activos en la zona, `lluvia` (mm), `zona` (centro, norte, sur) y el `tiempo` de entrega. Los pedidos aumentan cuando llueve. La **fila 0** es una entrega muy lejana y más rápida de lo esperado. Vas a ajustar el modelo múltiple en **forma matricial**, a diagnosticarlo y a ver qué pasa si se omite una variable.

**Enunciado**
1. `X`: matriz de diseño (`ndarray`, 150×7) con columnas en este orden: constante, `distancia`, `cuadras`, `pedidos`, `lluvia`, `zona_norte`, `zona_sur` (usa `pd.get_dummies(..., drop_first=True, dtype=float)`: la referencia es `centro`).
2. `mco(X, y)` → `(beta, residuos)` (con `np.linalg.lstsq`). Con ellos: `s2` $=SCE/(n-p)$ y `ee` (errores estándar $\sqrt{s^2\,[(X^\top X)^{-1}]_{jj}}$).
3. Matriz sombrero $H=X(X^\top X)^{-1}X^\top$: `h` (su diagonal, el apalancamiento), `traza` $=\sum h_{ii}$ y la **distancia de Cook** $D_i=\dfrac{e_i^2}{p\,s^2}\cdot\dfrac{h_{ii}}{(1-h_{ii})^2}$ (`cook`, vector) con `i_cook` (índice de la mayor).
4. `vif`: diccionario `{nombre: VIF}` para `distancia`, `cuadras`, `pedidos` y `lluvia`, donde $VIF_j=1/(1-R_j^2)$ y $R_j^2$ sale de regresar la columna $j$ sobre **todas las demás** de `X`. Luego `inflacion_ee`: error estándar de `distancia` con `cuadras` / error estándar de `distancia` **sin** `cuadras`.
5. **Variable omitida:** `b_lluvia_con` (coeficiente de `lluvia` en el modelo completo) y `b_lluvia_sin` (el mismo coeficiente tras **quitar `pedidos`**).
6. **$F$ parcial** para `zona` (¿aportan las dos indicadoras?): `F_zona` $=\dfrac{(SCE_r-SCE_c)/q}{SCE_c/(n-p)}$ con $q$ restricciones y `p_zona` (cola superior de $F_{q,\,n-p}$).

**Criterios de aceptación:** todo coincide con `statsmodels` (tolerancia $10^{-6}$ o mejor); `traza` es igual al número de parámetros; la observación influyente es la **fila 0**; los VIF de `distancia` y `cuadras` son enormes y los de `pedidos` y `lluvia` moderados; y `b_lluvia_sin` es **mayor** que `b_lluvia_con` (la lluvia carga con el efecto de los pedidos).

**Puntaje sugerido:** 22 puntos.

In [ ]:
entregas = generar_entregas()
yt = entregas["tiempo"].to_numpy()
n3 = len(entregas)
NUM = ["distancia", "cuadras", "pedidos", "lluvia"]


def mco(X, y):
    # TODO: devuelve (beta, residuos) usando np.linalg.lstsq
    return None


# TODO: completa
X = None                  # (150, 7): constante, distancia, cuadras, pedidos, lluvia, zona_norte, zona_sur
beta = e = None
p3 = s2_3 = None          # p3 = número de columnas de X; s2_3 = SCE / (n - p)
ee = None                 # errores estándar de beta
h = traza = None          # apalancamiento y su suma
cook = i_cook = None
vif = None                # {"distancia": ..., "cuadras": ..., "pedidos": ..., "lluvia": ...}
inflacion_ee = None
b_lluvia_con = b_lluvia_sin = None
F_zona = p_zona = None

In [ ]:
# Autoverificación del Reto 3 (no modificar)
assert X is not None and X.shape == (n3, 7) and np.allclose(X[:, 0], 1) and np.allclose(X[:, 1:5], entregas[NUM].to_numpy()), "X debe ser 150×7: columna de unos y luego distancia, cuadras, pedidos, lluvia."
assert np.allclose(X[:, 5], (entregas["zona"] == "norte")) and np.allclose(X[:, 6], (entregas["zona"] == "sur")), "Las dos últimas columnas son las indicadoras zona_norte y zona_sur (referencia: centro)."
_m = sm.OLS(yt, X).fit()
assert beta is not None and np.allclose(beta, _m.params, atol=1e-8) and np.allclose(e, _m.resid, atol=1e-8), "mco debe devolver (beta, residuos) = (lstsq(X, y), y - X·beta)."
assert p3 == 7 and abs(s2_3 - _m.scale) < 1e-8, "s2_3 = SCE / (n - p) con p = 7 parámetros."
assert ee is not None and np.allclose(ee, _m.bse, atol=1e-8), "ee = sqrt(s² · diag((XᵀX)⁻¹))."
_inf = _m.get_influence()
assert h is not None and np.allclose(h, _inf.hat_matrix_diag, atol=1e-8) and abs(traza - 7) < 1e-8, "h es la diagonal de X(XᵀX)⁻¹Xᵀ; su suma debe ser p = 7."
assert cook is not None and np.allclose(cook, _inf.cooks_distance[0], atol=1e-8), "Cook: e² / (p·s²) · h / (1 - h)²."
assert i_cook == 0 and cook[0] > 5 * np.sort(cook)[-2], f"La observación más influyente debe ser la fila 0 (la entrega lejana y rápida); tu i_cook = {i_cook}."
assert vif is not None and set(vif) == set(NUM), "vif debe ser un diccionario con las llaves distancia, cuadras, pedidos y lluvia."
for _j, _nom in enumerate(NUM, start=1):
    assert abs(vif[_nom] - variance_inflation_factor(X, _j)) < 1e-6, f"VIF de {_nom}: 1/(1 - R²) regresando esa columna sobre TODAS las demás de X (incluida la constante y las indicadoras)."
assert min(vif["distancia"], vif["cuadras"]) > 20 and max(vif["pedidos"], vif["lluvia"]) < 5, "Distancia y cuadras deben tener VIF enorme; pedidos y lluvia, moderado."
_Xs = np.delete(X, 2, axis=1)
_ms = sm.OLS(yt, _Xs).fit()
assert inflacion_ee is not None and abs(inflacion_ee - _m.bse[1] / _ms.bse[1]) < 1e-6 and inflacion_ee > 3, "inflacion_ee = ee de distancia (modelo completo) / ee de distancia (modelo sin cuadras, reajustado desde cero)."
assert abs(b_lluvia_con - _m.params[4]) < 1e-8 and abs(b_lluvia_sin - sm.OLS(yt, np.delete(X, 3, axis=1)).fit().params[3]) < 1e-8, "b_lluvia_sin: reajusta quitando la columna de pedidos y toma el coeficiente de lluvia (que ahora está en la posición 3)."
assert b_lluvia_sin > b_lluvia_con + 0.5, "Al omitir pedidos (que sube con la lluvia y alarga el tiempo) el coeficiente de lluvia debe INFLARSE."
_c = _m.compare_f_test(sm.OLS(yt, X[:, :5]).fit())
assert F_zona is not None and abs(F_zona - _c[0]) < 1e-6 and abs(p_zona - _c[1]) < 1e-8, "F = ((SCE_reducido - SCE_completo)/q) / (SCE_completo/(n - p)) con q = 2; p con stats.f.sf(F, 2, n - p)."
print(f"✅ Reto 3 superado: Cook máximo en la fila {i_cook} ({cook[i_cook]:.2f}) | VIF distancia {vif['distancia']:.0f}, cuadras {vif['cuadras']:.0f} → error estándar ×{inflacion_ee:.1f} | lluvia pasa de {b_lluvia_con:.2f} a {b_lluvia_sin:.2f} min/mm al omitir pedidos | F(zona) = {F_zona:.2f} (p = {p_zona:.4f})")

---
## Reto 4: Datos de personas, hallazgos falsos y decisiones justas ⚖️

**Contexto:** tres situaciones del mismo observatorio. (a) Quiere publicar un **registro de salud** (`registro`) quitando el nombre; los cuasi-identificadores son `edad`, `municipio` y `ocupacion`. (b) Una exploración probó **20 hipótesis** y entregó sus valores $p$ (`pvals`). (c) Un modelo de crédito decidió sobre dos grupos, A y B (`decisiones`: `pago` real y `aprobado`).

**Enunciado**
1. **Privacidad.** `k_min(df, qi)`: tamaño del grupo más pequeño que comparte los mismos valores de `qi`; `n_unicos`: personas únicas con `QI`. `generalizar(df, ancho)`: copia con la columna `edad_banda` $=\lfloor \texttt{edad}/\texttt{ancho}\rfloor\cdot\texttt{ancho}$. `suprimir(df, qi, k)`: elimina las filas cuyo grupo tenga menos de `k` personas. Con `QI_GEN` (edad en bandas de 10 años), obtén `k_gen` (el $k$ tras generalizar, sin suprimir), `df_k` (tras suprimir con $k=5$) y `retenido` (fracción de filas que sobreviven).
2. **Ruido.** `laplace_cuenta(c, eps, rng)`: una cuenta `c` (sensibilidad 1) protegida con ruido de Laplace de escala $1/\varepsilon$. Estima `err_medio` $=\{\varepsilon: \text{error absoluto medio}\}$ para $\varepsilon\in\{0.1, 1\}$ con 20 000 réplicas (`np.random.default_rng(0)`), y `eps_total` del presupuesto gastado por `CONSULTAS` (composición secuencial).
3. **Multiplicidad.** `bonferroni(p, alpha)`, `holm(p, alpha)` y `benjamini_hochberg(p, alpha)`: arreglos booleanos de hipótesis rechazadas (`ALFA = 0.05`). Guarda `n_crudo` (rechazos sin corregir) y `rechazos = {"bonf": ..., "holm": ..., "bh": ...}` (conteos).
4. **Equidad.** Con `decisiones`: `tasa_sel` (aprobación por grupo), `dif_paridad` $=$ tasa(A) $-$ tasa(B), `razon_impacto` $=$ min / max de las tasas, `tpr` (tasa de aprobación entre quienes **sí pagaron**, por grupo) y `dif_tpr` $=$ tpr(A) $-$ tpr(B).

**Criterios de aceptación:** al generalizar baja `n_unicos`, y tras suprimir $k\ge 5$ en todo el archivo conservando más del 60 % de las filas; el error medio del ruido es $\approx 1/\varepsilon$ (5 %); BH rechaza más que Holm y Holm al menos lo de Bonferroni, y todos coinciden con `multipletests`; y las métricas de equidad coinciden con un cálculo independiente (con estos datos se incumple la regla del 80 % de la razón de impacto).

**Puntaje sugerido:** 23 puntos.

In [ ]:
registro = generar_registro()
QI = ["edad", "municipio", "ocupacion"]
QI_GEN = ["edad_banda", "municipio", "ocupacion"]
CONSULTAS = [0.25, 0.25, 0.25, 0.25, 0.5]
pvals, ALFA = generar_pvalores(), 0.05
decisiones = generar_decisiones()


def k_min(df, qi):
    # TODO: tamaño del grupo más pequeño que comparte los valores de qi
    return None


def generalizar(df, ancho):
    # TODO: copia de df con la columna edad_banda = (edad // ancho) * ancho
    return None


def suprimir(df, qi, k):
    # TODO: devuelve las filas cuyo grupo (por qi) tiene al menos k personas
    return None


def laplace_cuenta(c, eps, rng):
    # TODO: c + ruido de Laplace de escala 1/eps
    return None


def bonferroni(p, alpha):
    # TODO: arreglo booleano de rechazos
    return None


def holm(p, alpha):
    # TODO: arreglo booleano de rechazos (procedimiento escalonado descendente)
    return None


def benjamini_hochberg(p, alpha):
    # TODO: arreglo booleano de rechazos (procedimiento escalonado ascendente, controla la FDR)
    return None


# TODO: completa
k_orig, n_unicos = None, None      # k y personas únicas con QI (sin generalizar)
k_gen = None                       # k tras generalizar edad en bandas de 10 (QI_GEN), sin suprimir
df_k, retenido = None, None        # tras suprimir grupos con menos de 5; fracción conservada
err_medio = None                   # {0.1: ..., 1: ...}
eps_total = None
n_crudo, rechazos = None, None     # {"bonf": ..., "holm": ..., "bh": ...}
tasa_sel = dif_paridad = razon_impacto = None
tpr = dif_tpr = None

In [ ]:
# Autoverificación del Reto 4 (no modificar)
_s = registro.groupby(QI).size()
assert k_orig == _s.min() and n_unicos == int((_s == 1).sum()) and n_unicos > 500, "k_min = tamaño del grupo más pequeño (groupby(qi).size().min()); n_unicos cuenta los grupos de tamaño 1."
_g = generalizar(registro, 10)
assert _g is not None and "edad_banda" in _g.columns and (_g["edad_banda"] == registro["edad"] // 10 * 10).all() and "edad_banda" not in registro.columns, "generalizar debe devolver una COPIA con edad_banda = (edad // ancho) * ancho (no modifiques `registro`)."
_s2 = _g.groupby(QI_GEN).size()
assert k_gen == _s2.min() and int((_s2 == 1).sum()) < n_unicos / 3, "k_gen es el k de QI_GEN tras generalizar; las personas únicas deben bajar a menos de un tercio."
assert df_k is not None and k_min(df_k, QI_GEN) >= 5 and len(df_k) == int(_s2[_s2 >= 5].sum()), "Tras suprimir, TODOS los grupos de QI_GEN deben tener al menos 5 personas, y solo se quitan los grupos pequeños completos."
assert abs(retenido - len(df_k) / len(registro)) < 1e-12 and 0.6 < retenido < 1, "retenido = len(df_k) / len(registro), entre 0.6 y 1."
assert len(suprimir(_g, QI_GEN, 1)) == len(registro), "suprimir con k = 1 no debe quitar ninguna fila."
_r = np.random.default_rng(1)
_x = np.array([laplace_cuenta(50, 0.5, _r) for _ in range(20000)])
assert abs(np.mean(np.abs(_x - 50)) - 2.0) < 0.1 and abs(np.mean(_x - 50)) < 0.1, "laplace_cuenta(c, eps, rng) = c + rng.laplace(0, 1/eps): con eps = 0.5 el error absoluto medio debe rondar 2 y el sesgo, 0."
assert err_medio is not None and set(err_medio) == {0.1, 1} and abs(err_medio[0.1] - 10) < 0.5 and abs(err_medio[1] - 1) < 0.05, "El error absoluto medio del mecanismo de Laplace es sensibilidad/eps: ≈ 10 con eps = 0.1 y ≈ 1 con eps = 1."
assert eps_total is not None and abs(eps_total - 1.5) < 1e-12, "Composición secuencial: los ε se SUMAN (0.25·4 + 0.5 = 1.5)."
_ps = np.array([0.001, 0.012, 0.02, 0.04, 0.3])
assert list(bonferroni(_ps, 0.05)) == [True, False, False, False, False] and list(holm(_ps, 0.05)) == [True, True, False, False, False], "Ejemplo de control: p = [0.001, 0.012, 0.02, 0.04, 0.3] con alfa 0.05 → Bonferroni rechaza 1 (umbral 0.01) y Holm rechaza 2 (umbrales 0.01, 0.0125, 0.0167...)."
_m = {"bonf": "bonferroni", "holm": "holm", "bh": "fdr_bh"}
_f = {"bonf": bonferroni, "holm": holm, "bh": benjamini_hochberg}
for _k in _m:
    _esp = multipletests(pvals, alpha=ALFA, method=_m[_k])[0]
    assert np.array_equal(np.asarray(_f[_k](pvals, ALFA), dtype=bool), _esp), f"{_k}: debe rechazar exactamente las mismas hipótesis que statsmodels multipletests(method='{_m[_k]}')."
assert n_crudo == int((pvals < ALFA).sum()) and rechazos == {k: int(multipletests(pvals, alpha=ALFA, method=v)[0].sum()) for k, v in _m.items()}, "n_crudo cuenta p < ALFA; rechazos usa los conteos de cada procedimiento."
assert n_crudo > rechazos["bh"] > rechazos["holm"] >= rechazos["bonf"], "Debe cumplirse: sin corregir rechazas más que con BH, BH más que Holm (BH controla la FDR, no la FWER) y Holm al menos lo de Bonferroni."
_t = decisiones.groupby("grupo")["aprobado"].mean()
_q = decisiones[decisiones["pago"] == 1].groupby("grupo")["aprobado"].mean()
assert tasa_sel is not None and abs(tasa_sel["A"] - _t["A"]) < 1e-12 and abs(tasa_sel["B"] - _t["B"]) < 1e-12, "tasa_sel: media de `aprobado` por grupo."
assert abs(dif_paridad - (_t["A"] - _t["B"])) < 1e-12 and abs(razon_impacto - _t.min() / _t.max()) < 1e-12, "dif_paridad = tasa(A) - tasa(B); razon_impacto = min(tasas) / max(tasas)."
assert tpr is not None and abs(tpr["A"] - _q["A"]) < 1e-12 and abs(tpr["B"] - _q["B"]) < 1e-12 and abs(dif_tpr - (_q["A"] - _q["B"])) < 1e-12, "tpr: media de `aprobado` por grupo SOLO entre quienes pagaron (pago == 1); dif_tpr = tpr(A) - tpr(B)."
assert razon_impacto < 0.8 and dif_tpr > 0.1, "Con estos datos la razón de impacto debe ser menor que 0.8 y la brecha de oportunidad (TPR) mayor que 0.10."
print(f"✅ Reto 4 superado: únicos {n_unicos} → k = {k_gen} con bandas y {retenido:.0%} de filas con k ≥ 5 | ruido medio {err_medio[0.1]:.1f} (ε = 0.1) y {err_medio[1]:.2f} (ε = 1) | sin corregir {n_crudo} hallazgos; Bonf {rechazos['bonf']}, Holm {rechazos['holm']}, BH {rechazos['bh']} | paridad {dif_paridad:+.3f}, impacto {razon_impacto:.2f}, TPR {dif_tpr:+.3f}")

---
## Reto 5: Conclusiones 📝

**Puntaje sugerido:** 15 puntos (sin código). Escribe **5 a 8 líneas** en la celda de abajo interpretando *tus* resultados, con tus cifras. Debe responder:

- Con `p_t_limpio`, `p_t_err`, `U_err` y `prob_sup`: ¿por qué un solo cronómetro mal digitado cambió la conclusión de la $t$ y no la de Mann-Whitney? ¿Qué *no* te dice el valor $p$ que sí te dice `prob_sup` (o $\varepsilon^2$)? ¿Qué le dirías a la alcaldía sobre las rutas?
- ¿Por qué `ic_pred` es mucho más ancho que `ic_media` aunque salgan del mismo modelo? ¿Qué significan `r_c < rho_c` para decidir entre Pearson y Spearman, y qué **no** prueba una correlación alta entre flujo y tiempo?
- ¿Qué pasó con el error estándar de `distancia` al incluir `cuadras` (`inflacion_ee`) y por qué eso **no** invalida las predicciones del modelo? ¿Qué le pasó al coeficiente de `lluvia` al omitir `pedidos` y qué dice eso de interpretar un coeficiente aislado? ¿Qué harías con la fila 0?
- ¿Por qué quitar el nombre no anonimiza (`n_unicos`, `k_gen`) y qué costó llegar a $k\ge5$ (`retenido`)? ¿Por qué repetir la misma consulta con ruido de Laplace deshace la protección?
- De los `n_crudo` hallazgos, ¿cuántos sobreviven a Holm y a BH y por qué esa diferencia importa? Con `razon_impacto` y `dif_tpr`: ¿el modelo de crédito es «justo»? ¿Qué definición de justicia estás usando y cuál sacrificarías?

---
#### ✍️ Tus conclusiones

_Escribe aquí tus 5-8 líneas. Reemplaza este texto por tu análisis._

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: rangos con empates, Mann-Whitney (asintótico y permutación), robustez y Kruskal-Wallis | 22 |
| Reto 2: regresión simple, SC, IC de la pendiente, banda de media frente a predicción y correlaciones | 18 |
| Reto 3: regresión múltiple matricial, Cook, VIF, variable omitida y $F$ parcial | 22 |
| Reto 4: $k$-anonimato, ruido de Laplace, correcciones por multiplicidad y métricas de equidad | 23 |
| Reto 5: conclusiones argumentadas con tus cifras (5-8 líneas) | 15 |
| **Total** | **100** |

### ✅ Checklist de entrega

- [ ] Reinicié el kernel y ejecuté *Run All* sin errores.
- [ ] No quedan `# TODO` ni variables en `None`.
- [ ] Las cuatro celdas de autoverificación imprimen ✅.
- [ ] Contrasté mis resultados con SciPy/`statsmodels` solo como verificación, no como sustituto.
- [ ] En las conclusiones distingo significancia estadística de relevancia práctica, y correlación de causalidad.
- [ ] Escribí mis conclusiones con cifras de mis propios resultados.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Asignatura: Estadística, Análisis y Representación de Datos</i>
  </p>
</div>